In [ ]:
%pip install sb3-contrib

In [ ]:
import os
import json
from stable_baselines3 import DQN

import cat_toy_env
import gym_agents
import cat_actions

import importlib
importlib.reload(cat_toy_env)
importlib.reload(gym_agents)
importlib.reload(cat_actions)

def get_config_path(model_id):
    # 設定ファイルのパスを生成
    return os.path.join('models', model_id, 'config.json')

def load_env_and_config(model_id):
    config_path = get_config_path(model_id)
    with open(config_path) as f:
        config = json.load(f)

    env_config = config['env']['kwargs']

    # chaser（Cat）
    chaser_class = gym_agents.gym_agents_mapping[config['chaser']['name']]
    chaser_kwargs = config['chaser'].get('kwargs', {}).copy()
    # cat_actionsの文字列リストをクラスインスタンスに変換
    if 'cat_actions' in chaser_kwargs:
        chaser_kwargs['cat_actions'] = [cat_actions.cat_actions_mapping[name]() for name in chaser_kwargs['cat_actions']]
    chaser = lambda: chaser_class(**chaser_kwargs)

    # runners
    runners = []
    for runner_name, runner_info in config['runners'].items():
        runner_class = gym_agents.gym_agents_mapping[runner_name]
        runner_kwargs = runner_info['kwargs']
        runner_kwargs['vel_seq_len'] = config['env']['vel_seq_len']  # config.jsonの値で上書き
        probability = runner_info.get('probability', 1.0)
        runners.append((lambda rc=runner_class, rk=runner_kwargs: rc(**rk), probability))

    # 環境構築
    key_wards = {
        **env_config,
        'chaser': chaser,
        'runners': runners,
    }
    env = cat_toy_env.CatToyEnv(**key_wards)
    return env, key_wards, config

def train(model_id, learning_steps=150_000, model_loaded=False):
    env, key_wards, config = load_env_and_config(model_id)
    model = DQN("MlpPolicy", env, verbose=1)
    total_timesteps = 0
    if model_loaded:
        model = DQN.load(f"models/{model_id}/cat")
        env, key_wards, config = load_env_and_config(model_id)
        model.set_env(env)  # 必要に応じて新しい環境をセット

        if 'total_timesteps' in config:
            total_timesteps = config['total_timesteps']
        else:
            total_timesteps = 0
    
    # 学習
    model.learn(total_timesteps=learning_steps, log_interval=4)
    model.save(f'models/{model_id}/cat')

    # 合計timestepsをconfig.jsonに保存
    total_timesteps += learning_steps
    config['total_timesteps'] = total_timesteps
    with open(get_config_path(model_id), 'w') as f:
        json.dump(config, f, indent=2)

## 学習

In [ ]:
train_model_ids = ["3", "4"]
for model_id in train_model_ids:
    print(f"Training model {model_id}...")
    train(model_id, learning_steps=200_000, model_loaded=False)
    print(f"Model {model_id} trained and saved.")
print("All models trained successfully.")

In [ ]:
import numpy as np

model_id = "3"

env, key_wards, config = load_env_and_config(model_id)
key_wards_eval = key_wards.copy()
key_wards_eval["render_mode"] = "human"

env_eval = cat_toy_env.CatToyEnv(**key_wards_eval)

model = DQN.load(f"models/{model_id}/cat")

obs, _ = env_eval.reset()
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, rewards, terminated, truncated, info = env_eval.step(action)
    done = terminated or truncated

In [ ]:
import torch
import dqn_onnx
importlib.reload(dqn_onnx)

def export_policy_to_onnx(model_id):
    model = DQN.load(f"models/{model_id}/cat")
    policy_net = dqn_onnx.DQNOnnx(model.policy)
    env_dummy = cat_toy_env.CatToyEnv(**key_wards_eval)
    obs_dim = env_dummy.observation_space.shape[0]
    obs = torch.randn(1, obs_dim).detach().cpu()

    torch.onnx.export(
        policy_net,
        (obs),
        f'models/{model_id}/policy.onnx',
        export_params=True,
        opset_version=17,
        input_names=['obs'],
        output_names=['option', 'action'],
        dynamic_axes={
            'obs': {0: 'batch_size'},
            'option': {0: 'batch_size'},
            'action': {0: 'batch_size'},
        },
        training=torch.onnx.TrainingMode.EVAL
    )


import shutil

def move_onnx(model_id, src_dir, dst_dir):
    os.makedirs(os.path.join(dst_dir, model_id), exist_ok=True)
    onnx_src = os.path.join(src_dir, model_id, 'policy.onnx')
    onnx_dst = os.path.join(dst_dir, model_id, 'policy.onnx')
    shutil.move(onnx_src, onnx_dst)

def extract_index_info(config_path):
    with open(config_path) as f:
        config = json.load(f)
    name = config.get('name', '')
    vel_seq_len = config.get('env', {}).get('vel_seq_len', '')
    description = config.get('description', '')
    secret_info = config.get('secret_info', '')
    return {
        'name': name,
        'vel_seq_len': vel_seq_len,
        'description': description,
        'secret_info': secret_info
    }

def update_models_index(model_id, config_path, index_path):
    info = extract_index_info(config_path)
    info['id'] = model_id
    if os.path.exists(index_path):
        with open(index_path) as f:
            index = json.load(f)
    else:
        index = []
    index = [item for item in index if item.get('id') != model_id]
    index.append(info)
    with open(index_path, 'w', encoding='utf-8') as f:
        json.dump(index, f, ensure_ascii=False, indent=2)


In [ ]:
export_model_ids = ["1", "2", "3", "4"]
for model_id in export_model_ids:
    print(f"Exporting model {model_id} to ONNX...")
    export_policy_to_onnx(model_id)
    src_dir = 'models'
    dst_dir = '../cat-game/public/models'
    index_path = os.path.join(dst_dir, 'models_index.json')
    move_onnx(model_id, src_dir, dst_dir)
    config_path = os.path.join(src_dir, model_id, 'config.json')
    update_models_index(model_id, config_path, index_path)
    print(f"Model {model_id} exported to ONNX successfully.")

print("All models exported to ONNX successfully.")

In [ ]:
for model_id in export_model_ids: